# RQ1 Lifecycle Order Table with Severity Distribution

This notebook rebuilds the cleaned lifecycle-order table and adds severity distribution for each lifecycle order type.

Rules:

- Use all rows with valid Fix, Release, and Disclosure dates.
- Exclude rows where `Release Date < Fix Date` as timestamp/tag-selection anomalies.
- Use `Report` as the first item for transparent fixes.
- Use `None` as the fourth item for silent fixes.
- Add severity distribution per row in the format: `Critical (x), High (x), Medium (x), Low (x)`.
- `Unknown` is included only if present for a lifecycle order type.


In [1]:
import pandas as pd
from pathlib import Path


In [2]:
FIX_RELEASE_PATH = Path('../../data/rq1/external_release_nvd/fix_releases_from_patch_data_local_server.csv')
NVD_PATH = Path('../../data/rq1/external_release_nvd/cve_NVD_disclosure_dates.csv')
LINKS_PATH = Path('../../data/rq1/corrected_resolved_links_v2.csv')
SEVERITY_PATH = Path('../../data/rq1/repo_with_severity.csv')

fix_release = pd.read_csv(FIX_RELEASE_PATH)
nvd = pd.read_csv(NVD_PATH)
links = pd.read_csv(LINKS_PATH)
severity = pd.read_csv(SEVERITY_PATH)

print('fix_release:', fix_release.shape, 'unique CVEs:', fix_release['CVE_ID'].nunique())
print('nvd:', nvd.shape, 'unique CVEs:', nvd['CVE_ID'].nunique())
print('links:', links.shape, 'unique CVEs:', links['CVE_ID'].nunique())
print('severity:', severity.shape, 'unique CVEs:', severity['CVE_ID'].nunique())
severity['Severity'].value_counts(dropna=False)


fix_release: (832, 6) unique CVEs: 832
nvd: (832, 2) unique CVEs: 832
links: (832, 5) unique CVEs: 832
severity: (832, 10) unique CVEs: 832


Severity
Medium      608
High        140
Low          47
Critical     36
Unknown       1
Name: count, dtype: int64

In [3]:
# Build event-level dataset.
df = fix_release[fix_release['Oldest Tag Date'].astype(str).str.lower().ne('not found')].copy()

df['Fix Date'] = pd.to_datetime(df['Commit Date'], errors='coerce')
df['Release Date'] = pd.to_datetime(df['Oldest Tag Date'], errors='coerce')

df = df.merge(nvd[['CVE_ID', 'Published Date']], on='CVE_ID', how='left')
df['Disclosure Date'] = pd.to_datetime(df['Published Date'], errors='coerce')

df = df.merge(links[['CVE_ID', 'Link Presence']], on='CVE_ID', how='left')
df['Reporting characteristics'] = df['Link Presence'].map({
    'contains links': 'Transparent',
    'no links': 'Silent',
})

df = df.merge(severity[['CVE_ID', 'Severity']], on='CVE_ID', how='left')

print('After requiring release date:', df.shape)
print('Missing Fix Date:', df['Fix Date'].isna().sum())
print('Missing Release Date:', df['Release Date'].isna().sum())
print('Missing Disclosure Date:', df['Disclosure Date'].isna().sum())
print('Missing Reporting characteristics:', df['Reporting characteristics'].isna().sum())
print('Missing Severity:', df['Severity'].isna().sum())


After requiring release date: (758, 13)
Missing Fix Date: 0
Missing Release Date: 0
Missing Disclosure Date: 0
Missing Reporting characteristics: 0
Missing Severity: 0


In [4]:
# Keep rows where all events and severity are available.
event_df = df.dropna(subset=[
    'Fix Date',
    'Release Date',
    'Disclosure Date',
    'Reporting characteristics',
    'Severity',
]).copy()

# Exclude rows where release appears before fix.
release_before_fix_df = event_df[event_df['Release Date'] < event_df['Fix Date']].copy()
event_df = event_df[event_df['Release Date'] >= event_df['Fix Date']].copy()

print('Rows excluded because Release Date < Fix Date:', release_before_fix_df.shape[0])
print('Rows available for lifecycle order + severity table:', event_df.shape[0])
print('Unique CVEs:', event_df['CVE_ID'].nunique())
event_df['Severity'].value_counts(dropna=False)


Rows excluded because Release Date < Fix Date: 15
Rows available for lifecycle order + severity table: 743
Unique CVEs: 743


Severity
Medium      542
High        126
Low          43
Critical     31
Unknown       1
Name: count, dtype: int64

In [5]:
def order_events(row):
    events = [
        ('Fix', row['Fix Date']),
        ('Release', row['Release Date']),
        ('Disclosure', row['Disclosure Date']),
    ]

    tie_breaker = {
        'Fix': 0,
        'Release': 1,
        'Disclosure': 2,
    }

    ordered = sorted(events, key=lambda item: (item[1], tie_breaker[item[0]]))
    ordered_names = [name for name, _ in ordered]

    if row['Reporting characteristics'] == 'Transparent':
        return ['Report'] + ordered_names

    return ordered_names + ['None']

ordered_columns = event_df.apply(order_events, axis=1, result_type='expand')
ordered_columns.columns = ['First', 'Second', 'Third', 'Fourth']

order_df = pd.concat([
    event_df[['CVE_ID', 'PATCH', 'Reporting characteristics', 'Severity', 'Fix Date', 'Release Date', 'Disclosure Date']].reset_index(drop=True),
    ordered_columns.reset_index(drop=True),
], axis=1)

order_df.head()


,CVE_ID,PATCH,Reporting characteristics,Severity,Fix Date,Release Date,Disclosure Date,First,Second,Third,Fourth
0,CVE-2013-4600,https://github.com/alkacon/opencms-core/commit...,Transparent,Medium,2013-06-20 15:32:02,2013-07-09 11:57:20,2013-08-09,Report,Fix,Release,Disclosure
1,CVE-2018-3831,https://github.com/elastic/elasticsearch/commi...,Transparent,Medium,2018-08-29 16:31:56,2018-08-29 16:31:56,2018-09-19,Report,Fix,Release,Disclosure
2,CVE-2013-4310,https://github.com/apache/struts/commit/0c8366...,Transparent,Medium,2013-10-15 18:20:43,2013-10-15 18:58:03,2013-09-30,Report,Disclosure,Fix,Release
3,CVE-2018-1000615,https://github.com/opennetworkinglab/onos/comm...,Silent,Medium,2018-06-27 21:07:37,2018-08-13 23:15:43,2018-07-09,Fix,Disclosure,Release,None
4,CVE-2021-39154,https://github.com/x-stream/xstream/commit/652...,Silent,Medium,2021-05-26 19:43:09,2021-08-22 13:57:44,2021-08-23,Fix,Release,Disclosure,None


In [6]:
severity_order = ['Critical', 'High', 'Medium', 'Low', 'Unknown']

def format_severity_distribution(values):
    counts = values.value_counts(dropna=False).to_dict()
    parts = []

    for severity_label in severity_order:
        count = int(counts.get(severity_label, 0))
        if severity_label == 'Unknown':
            if count > 0:
                parts.append(f'{severity_label} ({count})')
        else:
            parts.append(f'{severity_label} ({count})')

    # Include any unexpected severity labels at the end.
    for severity_label, count in sorted(counts.items()):
        if severity_label not in severity_order:
            parts.append(f'{severity_label} ({int(count)})')

    return ', '.join(parts)

summary_table = (
    order_df
    .groupby(['Reporting characteristics', 'First', 'Second', 'Third', 'Fourth'])
    .agg(
        Count=('CVE_ID', 'count'),
        **{'Severity distribution': ('Severity', format_severity_distribution)}
    )
    .reset_index()
)

summary_table['%'] = (
    summary_table['Count']
    / summary_table.groupby('Reporting characteristics')['Count'].transform('sum')
    * 100
).round(2)

summary_table = summary_table[
    ['Reporting characteristics', 'First', 'Second', 'Third', 'Fourth', 'Count', '%', 'Severity distribution']
].sort_values(
    ['Reporting characteristics', 'Count'],
    ascending=[True, False],
).reset_index(drop=True)

summary_table


,Reporting characteristics,First,Second,Third,Fourth,Count,%,Severity distribution
0,Silent,Fix,Release,Disclosure,None,258,81.90,"Critical (6), High (38), Medium (202), Low (11..."
1,Silent,Fix,Disclosure,Release,None,38,12.06,"Critical (1), High (7), Medium (28), Low (2)"
2,Silent,Disclosure,Fix,Release,None,19,6.03,"Critical (1), High (0), Medium (17), Low (1)"
3,Transparent,Report,Fix,Release,Disclosure,323,75.47,"Critical (18), High (57), Medium (226), Low (22)"
4,Transparent,Report,Fix,Disclosure,Release,77,17.99,"Critical (3), High (15), Medium (53), Low (6)"
5,Transparent,Report,Disclosure,Fix,Release,28,6.54,"Critical (2), High (9), Medium (16), Low (1)"


In [7]:
# Optional detail table: severity counts as separate columns.
severity_wide = (
    order_df
    .pivot_table(
        index=['Reporting characteristics', 'First', 'Second', 'Third', 'Fourth'],
        columns='Severity',
        values='CVE_ID',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

severity_wide


Severity,Reporting characteristics,First,Second,Third,Fourth,Critical,High,Low,Medium,Unknown
0,Silent,Disclosure,Fix,Release,None,1,0,1,17,0
1,Silent,Fix,Disclosure,Release,None,1,7,2,28,0
2,Silent,Fix,Release,Disclosure,None,6,38,11,202,1
3,Transparent,Report,Disclosure,Fix,Release,2,9,1,16,0
4,Transparent,Report,Fix,Disclosure,Release,3,15,6,53,0
5,Transparent,Report,Fix,Release,Disclosure,18,57,22,226,0


In [8]:
# Save nothing by default. Uncomment if needed.
# summary_table.to_csv('rq1_lifecycle_order_severity_table.csv', index=False)
